# LCO Queue Calendar Sync — Stage 3 Demo (sync_lco_observation_calendar)

This notebook demonstrates `solsys_code/management/commands/sync_lco_observation_calendar.py`
(issue #37 Stage 3), the LCO queue sync command that syncs FTS/MuSCAT4 LCO queue
`ObservationRecord`s to the FOMO calendar as `CalendarEvent`s — transitioning from
a `[QUEUED]` scheduling-window banner to a clean placed block as the LCO scheduler
acts, with no-churn idempotency on unchanged records.

It demonstrates:

- Building a fixture `ObservationRecord` matching the shape used by
  `solsys_code/tests/test_sync_lco_observation_calendar.py`
- Invoking `sync_lco_observation_calendar` via `call_command` while the record is
  still unscheduled (queue window banner, `[QUEUED]` title prefix)
- Inspecting the resulting `[QUEUED]` `CalendarEvent` (times from
  `parameters['start']`/`parameters['end']`, url from `LCOFacility().get_observation_url`)
- Simulating the LCO scheduler placing the observation (`scheduled_start`/`scheduled_end`
  populated) and re-running the command to see the event transition to a clean
  placed block
- Re-running the command a third time with no changes to confirm no-churn
  idempotency (`modified` timestamp untouched, `unchanged: 1` in the summary)

This notebook lives in `pre_executed/` because it is **DB-dependent** (it creates
an `ObservationRecord` fixture and `CalendarEvent` rows) and is therefore **NOT
run during Sphinx/CI/ReadTheDocs builds**, per `docs/notebooks/README.md`.


## Django setup

Standard boilerplate to make `src.fomo.settings` importable from this notebook's
location (`docs/notebooks/pre_executed/` — three levels under the repo root, so
`parents[2]` gives the repo root) and to allow synchronous ORM calls inside
Jupyter's async event loop.


In [ ]:
import os
import sys
from pathlib import Path

import django

# Ensure the repo root is on sys.path so `src.fomo.settings` is importable
# when this notebook is executed from docs/notebooks/pre_executed/.
# NOTE: parents[2] is correct only when the Jupyter kernel CWD is
# docs/notebooks/pre_executed/. Start Jupyter from that directory, or
# adjust the index if you launch from the repo root.
repo_root_path = Path.cwd().resolve().parents[2]
assert (repo_root_path / 'manage.py').exists(), (
    f'Repo root not found at {repo_root_path}. '
    'Run Jupyter from docs/notebooks/pre_executed/ or adjust parents[] index.'
)
repo_root = str(repo_root_path)
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'src.fomo.settings')

# Jupyter's ipykernel runs inside an asyncio event loop, but Django's ORM is
# sync-only by default and refuses to run there; this opts back in.
os.environ.setdefault('DJANGO_ALLOW_ASYNC_UNSAFE', 'true')

django.setup()

## Create a fixture ObservationRecord

`sync_lco_observation_calendar` queries
`ObservationRecord(facility='LCO', parameters__proposal=<code>)`. Here we build one
fixture record using the same `parameters` shape as
`solsys_code/tests/test_sync_lco_observation_calendar.py` (`proposal`, `start`,
`end`, `instrument_type`, `site` keys). The FK setup — a non-sidereal `Target` via
`NonSiderealTargetFactory` (FOMO targets are Solar System minor bodies, not
sidereal sources) and a `user` — mirrors that test's `setUpTestData`.


In [ ]:
from django.contrib.auth import get_user_model
from tom_observations.models import ObservationRecord
from tom_targets.models import Target
from tom_targets.tests.factories import NonSiderealTargetFactory

DEMO_PROPOSAL = 'DEMOCODE'
DEMO_TARGET_NAME = 'sync-demo-target'

# get_or_create / fixed-name lookup keep this cell re-runnable without
# accumulating a new Target (and its factory-generated extras/aliases) on
# every re-run. NonSiderealTargetFactory is used because FOMO targets are
# Solar System minor bodies (orbital elements), not sidereal sources.
demo_user, _ = get_user_model().objects.get_or_create(username='sync-demo-user')
demo_target = Target.objects.filter(name=DEMO_TARGET_NAME).first()
if demo_target is None:
    demo_target = NonSiderealTargetFactory.create(name=DEMO_TARGET_NAME)

record, created = ObservationRecord.objects.get_or_create(
    observation_id='demo-900001',
    defaults=dict(
        target=demo_target,
        user=demo_user,
        facility='LCO',
        status='PENDING',
        scheduled_start=None,
        scheduled_end=None,
        parameters={
            'proposal': DEMO_PROPOSAL,
            'start': '2026-07-01T00:00:00',
            'end': '2026-07-02T00:00:00',
            'instrument_type': '2M0-SCICAM-MUSCAT',
            'site': 'coj',
        },
    ),
)

print('observation_id  :', record.observation_id)
print('status          :', record.status)
print('scheduled_start :', record.scheduled_start)
print('parameters      :', record.parameters)

In [ ]:
from django.forms import model_to_dict
import pprint

obsrecs = ObservationRecord.objects.all()
pprint.pprint(model_to_dict(obsrecs[0]))

## Sync the queued (unscheduled) record

`call_command` is the Django-recommended way to invoke management commands
programmatically. With `scheduled_start=None`, the command derives its times from
`parameters['start']`/`parameters['end']` and prefixes the title with `[QUEUED]`
(SYNC-02). The event's `url` is built via
`LCOFacility().get_observation_url(observation_id)` (SYNC-01).


In [ ]:
import io

from django.core.management import call_command

stdout_buf = io.StringIO()
stderr_buf = io.StringIO()

call_command('sync_lco_observation_calendar', '--proposal', DEMO_PROPOSAL, stdout=stdout_buf, stderr=stderr_buf)

print('stdout:', stdout_buf.getvalue())
if stderr_buf.getvalue():
    print('stderr:', stderr_buf.getvalue())

## Inspect the queued CalendarEvent

The resulting event has a `[QUEUED]`-prefixed title, times taken from the
parameters window, and a url built from the LCO portal (containing `/requests/`).


In [ ]:
from tom_calendar.models import CalendarEvent
from tom_observations.facilities.lco import LCOFacility

event_url = LCOFacility().get_observation_url('demo-900001')
event = CalendarEvent.objects.get(url=event_url)

print('title       :', event.title)  # starts with '[QUEUED]'
print('start_time  :', event.start_time.isoformat())
print('end_time    :', event.end_time.isoformat())
print('url         :', event.url)  # contains '/requests/'
print('telescope   :', event.telescope)
print('instrument  :', event.instrument)
print('proposal    :', event.proposal)
print('description :')
for line in event.description.splitlines():
    print(' ', line)

## Check all other CalendarEvent's


In [ ]:
events = CalendarEvent.objects.all()
for event in events:
    print(
        f'{event.title} {event.start_time.isoformat()}->{event.end_time.isoformat()} {event.telescope}+{event.instrument} {event.proposal}'
    )

## Simulate the LCO scheduler placing the observation

Setting `scheduled_start`/`scheduled_end` on the record and re-running the command
transitions the event to the clean placed form — times now come from the
scheduled fields and the `[QUEUED]` prefix drops (SYNC-03).


In [ ]:
from datetime import datetime
from datetime import timezone as dt_timezone

record.scheduled_start = datetime(2026, 7, 5, 10, 0, 0, tzinfo=dt_timezone.utc)
record.scheduled_end = datetime(2026, 7, 5, 12, 0, 0, tzinfo=dt_timezone.utc)
record.save()

call_command(
    'sync_lco_observation_calendar',
    '--proposal',
    DEMO_PROPOSAL,
    stdout=io.StringIO(),
    stderr=io.StringIO(),
)

event = CalendarEvent.objects.get(url=event_url)
print('title      :', event.title)  # no '[QUEUED]' prefix now
print('start_time :', event.start_time.isoformat())
print('end_time   :', event.end_time.isoformat())

## No-churn idempotency

Re-running the command a third time with no changes leaves the event's `modified`
timestamp unchanged, and the stdout summary reports `unchanged: 1` (SYNC-04).


In [ ]:
modified_before = CalendarEvent.objects.get(url=event_url).modified

stdout_buf3 = io.StringIO()
call_command(
    'sync_lco_observation_calendar',
    '--proposal',
    DEMO_PROPOSAL,
    stdout=stdout_buf3,
    stderr=io.StringIO(),
)

event = CalendarEvent.objects.get(url=event_url)
print('modified unchanged:', event.modified == modified_before)
print('stdout:', stdout_buf3.getvalue())  # shows 'unchanged: 1'

## Summary

This notebook demonstrates the Phase 04 `sync_lco_observation_calendar` command
satisfying requirements SELECT-01, SYNC-01, SYNC-02, SYNC-03, SYNC-04, and TERM-01:

| Requirement | Description | Demonstrated by |
|-------------|-------------|------------------|
| SELECT-01 | Only `ObservationRecord`s matching the given `--proposal` are synced | `--proposal DEMOCODE` selects the fixture record |
| SYNC-01 | Event `url` keyed on `LCOFacility().get_observation_url(observation_id)`, containing `/requests/` | url printed and used as the lookup key throughout |
| SYNC-02 | Unscheduled record (`scheduled_start=None`) uses `parameters['start']`/`['end']` and gets a `[QUEUED]` title prefix | first sync, "Inspect the queued CalendarEvent" |
| SYNC-03 | Scheduled record (`scheduled_start`/`scheduled_end` populated) uses those times and a clean title (no `[QUEUED]`) | "Simulate the LCO scheduler placing the observation" |
| SYNC-04 | Re-running with no changes leaves `modified` untouched and reports `unchanged: 1` | "No-churn idempotency" |
| TERM-01 | Terminal failure statuses get a `[EXPIRED]`/`[CANCELLED]`/`[FAILED]` title prefix | not exercised by this fixture; see `solsys_code/tests/test_sync_lco_observation_calendar.py` for the terminal-state test cases |

The command is invoked via `call_command` — equivalent to running
`./manage.py sync_lco_observation_calendar --proposal <code>` from the shell.

This notebook is **pre-executed** and intentionally excluded from automated doc
builds (not referenced in `docs/notebooks.rst`) because it depends on a live
`ObservationRecord`/`CalendarEvent` fixture in the local dev DB.


## Teardown

This notebook is DB-dependent and creates real rows in the local dev
database. Clean up everything created above so re-running the notebook
(or other notebooks/tests sharing the dev DB) starts from a clean slate:
the `CalendarEvent`, the `ObservationRecord`, the demo `Target` (whose
factory-generated `TargetExtra`/`TargetName` rows cascade-delete with it),
and the demo `User`.

Deletion order matters: `ObservationRecord.user` is `on_delete=DO_NOTHING`,
so the record must be deleted before the user. `CalendarEvent` has no FK
to either `ObservationRecord` or `Target` (it's matched by `url` in the
sync command), so it must be deleted explicitly rather than relying on
cascade.


In [ ]:
CalendarEvent.objects.filter(url=event_url).delete()
ObservationRecord.objects.filter(observation_id='demo-900001').delete()
Target.objects.filter(name=DEMO_TARGET_NAME).delete()
get_user_model().objects.filter(username='sync-demo-user').delete()

print('Teardown complete: demo CalendarEvent, ObservationRecord, Target, and User removed.')